In [1]:
import numpy as np 
import pandas as pd 
import os
import math 
import geopandas as gpd 


/opt/conda/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [3]:
path = "../../data/data_biaised/hegp_data_w_biais/"
# dir_list = os.listdir(path)
data_np = {}
data = {}

for x in os.listdir(path):
    if x.endswith("np.csv"):
        # Prints only text file present in My Folder
        data_np[x] = pd.read_csv(path+x, sep=";")
    else : 
        data[x] = pd.read_csv(path+x, sep=";")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/data_biaised/hegp_data_w_biais/'

In [ ]:
df_revenus = pd.read_csv("../analyse_clinique/insee_revenu/BASE_TD_FILO_DISP_IRIS_2020.csv",sep=";",dtype=str)[["IRIS","DISP_MED20"]]
df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')


## Analyse de l'impact du biais sur nos références

In [ ]:
df_revenus[df_revenus["DISP_MED20"]=="ns"] = np.nan
df_revenus[df_revenus["DISP_MED20"]=="nd"] = np.nan

df_revenus["DISP_MED20"] = df_revenus["DISP_MED20"].astype(float)

df_revenus = df_revenus.rename({"IRIS":"CODE_IRIS"},axis=1)


In [ ]:
## Distance entre deux point d'une sphère => coordonnées exprimées en WGS84 (degrès décimaux): comprend un modèle de la terre  d'où le calcul de distance angulaire 
def calculer_distance_haversine(lat1, lon1, lat2, lon2):

    try : 
        # Rayon de la Terre en kilomètres
        R = 6371.0

        # Conversion des degrés en radians
        dLat = math.radians(lat2 - lat1)
        dLon = math.radians(lon2 - lon1)
        rLat1 = math.radians(lat1)
        rLat2 = math.radians(lat2)

        # Formule de Haversine
        a = math.sin(dLat / 2)**2 + math.cos(rLat1) * math.cos(rLat2) * math.sin(dLon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        # Distance totale
        distance = R * c
        return distance
    except Exception as e : 
        print(f"Erreur lors du calcul de la distance : {e}")
        return np.nan
    

In [ ]:
## X,Y de référence : 'latitude','longitude'
## X,Y à évaluer : 'x','y'
data_w_info = {}
data = data_np
for f in data: 
    df= data[f]
    tag = f.split('_')[2].split('.')[0]
    df['tag'] = tag 

    ##-----------------------------------
    ## Ajout CODE IRIS initial 

    df = df.rename({'CODE_IRIS':'CODE_IRIS_geocoded'},axis=1)

    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude)).set_crs(epsg=4326)).to_crs(epsg=2154)
    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)
    df = df_join_iris.rename({'CODE_IRIS':'CODE_IRIS_init'},axis=1)

    ## ----------------------------------
    ## Ajout DISP_MED20 pour le code iris geocodé et le code iris initial 
    
    # Jointure pour récupérer le revenu médian associé à l'iris initial : 
    df = df.dropna(subset="CODE_IRIS_geocoded")
    df_rev_geoc = df.merge(df_revenus,how="left",left_on="CODE_IRIS_geocoded",right_on="CODE_IRIS")
    df_rev_geoc = df_rev_geoc.rename({'DISP_MED20':'DISP_MED20_geocoded'},axis=1)
    df_rev_geoc = df_rev_geoc.drop('CODE_IRIS',axis=1)


    # Jointure pour récupérer le revenu médian associé à l'iris géocodé : 
    df_rev_init = df_rev_geoc.dropna(subset="CODE_IRIS_init")
    df_rev_init = df_rev_init.merge(df_revenus,how="left",left_on="CODE_IRIS_init",right_on="CODE_IRIS")
    df_rev_init = df_rev_init.rename({'DISP_MED20':'DISP_MED20_init'},axis=1)
    
    df_rev = df_rev_init.copy()

    df_rev["DISP_MED20_geocoded"] = df_rev["DISP_MED20_geocoded"].astype(float)
    df_rev["DISP_MED20_init"] = df_rev["DISP_MED20_init"].astype(float)

    ## ----------------------------------
    ## Calcul distance et diff revenu 

    df_rev["y"] = df_rev["y"].astype(float)
    df_rev["x"] = df_rev["x"].astype(float)

    df_rev["latitude"] = df_rev["latitude"].astype(float)
    df_rev["longitude"] = df_rev["longitude"].astype(float)
    liste_ini =[]
    liste_geoc = [] 
    for i in df_rev.index : 

        lon1 = df_rev.at[i,"x"]
        lat1 = df_rev.at[i,"y"]
        lon2 = df_rev.at[i,"longitude"]
        lat2 = df_rev.at[i,"latitude"]

        # if pd.isna(lon1) or pd.isna(lat1): 
        #     liste_geoc.append(i)
        # if pd.isna(lon2) or pd.isna(lat2) :
        #     liste_ini.append(i)

        df_rev.loc[i,"distance_km"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)
        df_rev.loc[i,"distance_m"] = df_rev.loc[i,'distance_km']*1000

        ## ----------------------------------
        df_rev.loc[i,"diff_revenu"] =pd.to_numeric(df_rev.at[i,"DISP_MED20_init"]) - pd.to_numeric(df_rev.at[i,"DISP_MED20_geocoded"])


    data_w_info[tag] = df_rev

# file = 'df_ref_RETRAITE.csv'
# file.split('_')[2].split('.')[0]

In [ ]:
df_all = pd.concat(data_w_info,axis=0).reset_index()


In [ ]:

df_all['distance_m'] = df_all['distance_m'].astype(float)
df_all['diff_revenu'] = df_all['diff_revenu'].astype(float)
df_mean = df_all.groupby('tag')[['distance_m','diff_revenu']].mean()
df_med = df_all.groupby('tag')[['distance_m','diff_revenu']].median()

## % de changement d'iris par biais 

In [ ]:
all_tag = []
for tag in data_w_info: 
    all_tag.append(tag)

prc_chgmt_iris = {}
for tag in all_tag: 
    nb_changement = 0
    for i in df_all[df_all['tag']==tag].index:
        if df_all.loc[i,'CODE_IRIS_init'] != df_all.loc[i,'CODE_IRIS_geocoded']:
            nb_changement+=1
    prc_chgmt_iris[tag] = (nb_changement/len(df_all[df_all['tag']==tag]))*100

df_shift = pd.DataFrame.from_dict(prc_chgmt_iris, orient="index")
df_shift = df_shift.rename(columns={ 
    df_shift.columns[0]: " % de changement d'iris "
})


In [ ]:
df_final = pd.concat([df_med,df_shift],axis=1)

In [ ]:
df_final

### Visu distance 

In [ ]:
df_all[(df_all['numero_uai']=="0010516F")][['numero_uai','adresse','requete','distance_km']]
# (df_all['distance_km']>10) & 

In [ ]:
test = df_all[df_all['distance_km']>5]
# test = df_all.groupby('numero_uai')['distance_m']
# test[test['distance_m']>] 
# df_count = pd.DataFrame(test.groupby('numero_uai').size())
# print(len(df_count))
# df_count = df_count.rename({'0':'count'},axis=1).reset_index()
# df_count.columns
# res = df_count.groupby(0).size()
# res 
# df_mean = df_all.groupby('tag')[['distance_m','diff_revenu']].mean()



df_count = pd.DataFrame(test.groupby('tag').size())
print(len(df_count))
df_count = df_count.rename({'0':'count'},axis=1).reset_index()
# res = df_count.groupby(0).size()
# res 
df_count

In [ ]:
import matplotlib.pyplot as plt 
import plotly.express as px


# fig = px.histogram(df_all, x="distance_m")
# fig.show()

fig = px.violin(df_all[df_all['distance_km']<0.3], y="distance_m", box=True, color="tag", # draw box plot inside the violin
                # points='all', # can be 'outliers', or False
               )
fig.show()

In [ ]:
df_all[df_all['distance_km']>200]

In [ ]:
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

long_df = pd.melt(df_all, value_vars=['DISP_MED20_init', 'DISP_MED20_geocoded'], var_name='Type', value_name='Revenu')

# fig = px.density_contour(long_df, x='Revenu', y='Type', marginal_x='histogram', color='Type',
#                          title='Comparaison des distributions de revenu: INIT vs GEOCODED')

# fig.show()

# positions = pd.melt(df, id_vars=['adresse'], value_vars=['Numbers','Street type','Zipcode','City','Add. info'], 
#                     var_name='Type', value_name='Position_relative')


# Graphique de densité corrigé
sns.kdeplot(data=long_df, x='Revenu', hue='Type', fill=True, common_norm=False, alpha=0.5)
plt.title('Density comparison of median income for our reference')#Densité de la position relative des éléments de l\'adresse')
plt.xlabel('Revenu')
plt.ylabel('Density')
plt.show()


In [ ]:
fig = px.scatter(df_all[df_all['distance_km']<10],x='distance_km',y='diff_revenu')
fig.show()